# EBI BioImage Archive — Update Notebook

**Dataset:** 2026 Multicolour Bayer-SMLM paper (accession `S-BIAD3210`)
**Source:** SMB share on intelliflash
**Destination:** `ftp-private.ebi.ac.uk` (BioImage Archive Webin FTP)

## Why this notebook exists (not just re-running `EBI_Upload_Notebook.ipynb`)

The original April 2026 submission (`EBI_Upload_Notebook.ipynb`, §2) was a **selective** upload —
1,342 files under `Ximea/`, `ThorLabs/`, and two `ZWO/` folders, not the full `Data/` tree.
The real, correct record of what is *supposed* to be on EBI lives in
`EBI_Upload/2026_Multicolour_ebi_filelist.tsv` on the SMB share (1,342 rows, ~1.74 TB,
matches the `n_files`/`total_bytes` already recorded in `biostudies_checksums.json`).

Re-running §2 of the original notebook walks the *entire* `Data/` tree (every camera, every
folder ever acquired there) and compares that against the real ~1,342-file remote — which is
where the original "~1,300 files fewer on the EBI" mismatch came from. That part isn't missing
data; it's comparing the wrong baseline.

**However:** a live FTP listing against the real 1,342-file baseline itself still shows a
further few hundred files genuinely missing from the remote — evidently uploads that never
completed back in April (interrupted transfers, timeouts on the larger multi-GB files, etc.),
not a baseline-selection problem. §2 below finds exactly which baseline files those are and
re-uploads them **before** anything new is added, so the new-folder step starts from a remote
that's known to actually match the baseline.

A second, unrelated bug on 2026-08-28 compounded the original mismatch: an incremental-append
cell wrote to a *different* filename (`ebi_filelist.tsv`, no `2026_Multicolour_` prefix) than
the real baseline, so it silently dropped the 1,342 baseline rows and produced an 83-row
manifest containing only the two new folders. `biostudies_checksums.json`/
`2026_Multicolour_Filelist.json` on the share currently still reflect the correct 1,342-file
baseline (un-rebuilt since) — this notebook is what actually reconciles and extends it.

**Stale files left behind by that run** (safe to delete once this notebook's output is
verified): `EBI_Upload/ebi_filelist.tsv`, `EBI_Upload/ebi_upload_paths.txt`.

## What this notebook does, in order

1. **Load the real baseline** — read `2026_Multicolour_ebi_filelist.tsv` as ground truth for
   "supposed to be on EBI".
2. **Reconcile** — get the *actual* list of files present on the FTP remote (not just a count),
   diff it against the baseline, and upload whatever's missing. Re-verify until the remote
   genuinely matches the baseline.
3. **Hash the two new folders** — `ZWO/20260623_MASSIVECELLS` (Massive Cells, ZWO) and
   `ZWO/20260624_SAureus_NR4A` (S. aureus, ZWO) — the only folders not yet in the baseline.
4. **Merge and rewrite** the manifest (baseline + new rows) in place, backing up the original
   first.
5. **Upload** the two new folders' files over FTP.
6. **Verify** the remote count now matches the fully merged manifest.
7. **Rebuild and re-upload** `biostudies_checksums.json`.
8. **Rebuild** `2026_Multicolour_Filelist.json` for manual submission through the BioImage
   Archive portal.

> **Credentials:** enter Webin username/password via env vars if you don't want the
> hardcoded defaults below. Never hard-code or commit credentials elsewhere.


---
## 0 — Imports


In [ ]:
import csv
import ftplib
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from collections import defaultdict
from datetime import timedelta
from pathlib import Path

try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm not available — progress will be printed every 50 files")

print("Python:", sys.version)


---
## 1 — Configuration


In [ ]:
# ── Mount the SMB share (idempotent — safe to re-run) ─────────────────────────
SMB_SERVER = "intelliflash-mgmt-b.ch.private.cam.ac.uk"
SMB_SHARE  = "sycamore_asap_server"
SMB_URI    = f"smb://{SMB_SERVER}/{SMB_SHARE}/"


def ensure_smb_mounted(uri: str = SMB_URI) -> None:
    """Mount *uri* via GVFS (gio) unless it's already mounted."""
    listing = subprocess.run(["gio", "mount", "--list"], capture_output=True, text=True)
    if uri.rstrip("/") in listing.stdout:
        print(f"Already mounted: {uri}")
        return
    print(f"Mounting {uri} ...")
    result = subprocess.run(["gio", "mount", uri], capture_output=True, text=True)
    if result.returncode == 0 or "already mounted" in result.stderr.lower():
        print("Mounted.")
    else:
        print(result.stderr.strip())
        raise RuntimeError(
            f"Could not mount {uri} automatically -- gio may need an interactive "
            f'credential prompt; try running `gio mount "{uri}"` directly in a '
            "terminal first, then re-run this cell."
        )


def gvfs_path(server: str, share: str, subpath: str = "") -> Path:
    """Local filesystem path GVFS exposes an SMB mount at, once mounted."""
    base = Path(f"/run/user/{os.getuid()}/gvfs/smb-share:server={server},share={share}")
    return (base / subpath) if subpath else base


ensure_smb_mounted()


In [ ]:
# ── Source (SMB share) ────────────────────────────────────────────────────────
_gvfs_root   = gvfs_path(SMB_SERVER, SMB_SHARE, "2026_Multicolour_Paper/Data")
_legacy_root = Path('/scratch/sycamore-asap/2026_Multicolour_Paper/Data')
SMB_ROOT     = _gvfs_root if _gvfs_root.exists() else _legacy_root

# ── Output directory (EBI_Upload/ on the SMB share) ───────────────────────────
_gvfs_out   = gvfs_path(SMB_SERVER, SMB_SHARE, "2026_Multicolour_Paper/EBI_Upload")
_legacy_out = Path("/scratch/sycamore-asap/2026_Multicolour_Paper/EBI_Upload")
OUT_DIR     = (_gvfs_out if _gvfs_out.exists() else _legacy_out).resolve()

# ── The real baseline manifest — what is supposed to be on EBI ───────────────
BASELINE_TSV = OUT_DIR / "2026_Multicolour_ebi_filelist.tsv"

# ── Session outputs ────────────────────────────────────────────────────────────
RECONCILE_UPLOAD_PAIRS = OUT_DIR / "reconcile_missing_baseline_upload_paths.txt"
NEW_UPLOAD_PAIRS        = OUT_DIR / "pending_new_folders_upload_paths.txt"

# ── Portal/EBI-facing artefacts, rebuilt from the merged manifest ────────────
CHECKSUMS_JSON = OUT_DIR / "biostudies_checksums.json"
FILELIST_JSON  = OUT_DIR / "2026_Multicolour_Filelist.json"

# ── The two folders being added in this update ────────────────────────────────
NEW_FOLDERS = [
    "ZWO/20260623_MASSIVECELLS",   # Massive Cells, ZWO camera
    "ZWO/20260624_SAureus_NR4A",   # S. aureus / NR4A, ZWO camera
]

# ── EBI path prefix (preserved inside the submission) ─────────────────────────
EBI_PREFIX = "2026_Multicolour_Paper"
ACCESSION  = "S-BIAD3210"

# ── FTP credentials (EBI-issued) ───────────────────────────────────────────────
# WARNING: credentials are stored here — this notebook is gitignored.
# Override via env vars if preferred:
#   export WEBIN_USER="bs-upload"
#   export WEBIN_PASS="your_password"
FTP_HOST       = "ftp-private.ebi.ac.uk"
FTP_USER       = os.environ.get("WEBIN_USER", "bs-upload")
FTP_PASS       = os.environ.get("WEBIN_PASS", "vsr5nW7Y")
FTP_SECRET_DIR = "/dc/7de16d-c8a7-4bd3-b500-c664ccf4d3ba-a27000"
FTP_ROOT       = FTP_SECRET_DIR

# ── Sanity check ────────────────────────────────────────────────────────────
if SMB_ROOT.exists():
    print(f"SMB share found : {SMB_ROOT}")
else:
    print(f"WARNING: SMB share not found at {SMB_ROOT}")

if BASELINE_TSV.exists():
    print(f"Baseline manifest found : {BASELINE_TSV}")
else:
    print(f"WARNING: baseline manifest not found at {BASELINE_TSV}")
    print("Nothing to reconcile/merge onto — check OUT_DIR / the SMB mount before continuing.")

print(f"Output directory : {OUT_DIR}")
print(f"FTP host         : {FTP_HOST}")
print(f"FTP user         : {FTP_USER}")
print(f"FTP root         : {FTP_ROOT}")
print(f"New folders      : {NEW_FOLDERS}")


---
## Helper functions

Shared by the sections below — hashing, size formatting, and FTP plumbing, matching
`EBI_Upload_Notebook.ipynb`'s behaviour. `list_remote_files` is new here: unlike a plain
recursive count, it returns the actual set of remote paths so a specific missing file can be
identified and re-uploaded, not just noticed.


In [ ]:
def md5sum(path: Path, chunk: int = 1 << 20) -> str:
    """Return hex MD5 of a file, reading in 1 MB chunks."""
    h = hashlib.md5()
    with path.open("rb") as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()


def human_size(n_bytes: int) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} PB"


def remote_size(ftp: ftplib.FTP, path: str) -> int:
    """Return remote file size, or -1 if the file does not exist."""
    try:
        return ftp.size(path)
    except ftplib.error_perm:
        return -1


def ensure_remote_dirs(ftp: ftplib.FTP, remote_path: str):
    """Recursively create all intermediate directories on the FTP server."""
    parts = Path(remote_path).parent.parts
    for i in range(1, len(parts) + 1):
        d = str(Path(*parts[:i]))
        try:
            ftp.mkd(d)
        except ftplib.error_perm:
            pass  # already exists — fine


def upload_file(ftp: ftplib.FTP, local: Path, remote: str, chunk: int = 1 << 20) -> str:
    """Upload local → remote, skipping if sizes match. Returns 'skip'/'ok'/'error'."""
    local_size = local.stat().st_size
    if remote_size(ftp, remote) == local_size:
        return "skip"
    ensure_remote_dirs(ftp, remote)
    with local.open("rb") as f:
        ftp.storbinary(f"STOR {remote}", f, blocksize=chunk)
    return "ok"


def list_remote_files(ftp: ftplib.FTP, remote_dir: str) -> set:
    """Recursively list every file under remote_dir via NLST, returning full
    remote paths (not just a count) so specific missing files can be found."""
    files = set()
    try:
        entries = ftp.nlst(remote_dir)
    except ftplib.error_perm:
        return files
    for entry in entries:
        if Path(entry).suffix:  # has an extension → a file
            files.add(entry)
        else:  # directory — recurse
            files |= list_remote_files(ftp, entry)
    return files


def remote_file_set(ftp_host: str, ftp_user: str, ftp_pass: str,
                     ftp_root: str, ebi_prefix: str) -> set:
    """Connect, list everything under ftp_root/ebi_prefix, and return it as a set
    of manifest-style relative paths (i.e. starting with ebi_prefix/...)."""
    ftp = ftplib.FTP(ftp_host, timeout=60)
    ftp.login(ftp_user, ftp_pass)
    ftp.set_pasv(True)
    remote_root = f"{ftp_root}/"
    raw = list_remote_files(ftp, f"{ftp_root}/{ebi_prefix}")
    ftp.quit()
    return {p[len(remote_root):] if p.startswith(remote_root) else p for p in raw}


def load_upload_pairs(path: Path) -> list:
    pairs = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            src, dst = line.split("\t", 1)
            pairs.append((Path(src), dst))
    return pairs


def run_upload(ftp_host: str, ftp_user: str, ftp_pass: str,
               ftp_root: str, pairs_file: Path):
    if not pairs_file.exists():
        print(f"Path-pairs file not found: {pairs_file}")
        return

    pairs = load_upload_pairs(pairs_file)
    n     = len(pairs)
    print(f"Uploading {n} files to {ftp_host} …")

    ftp = ftplib.FTP(ftp_host, timeout=60)
    ftp.login(ftp_user, ftp_pass)
    ftp.set_pasv(True)
    print("Login OK")

    counts = {"ok": 0, "skip": 0, "error": 0}
    t0     = time.time()

    iterator = tqdm(enumerate(pairs, 1), total=n, unit="file") if HAS_TQDM \
               else enumerate(pairs, 1)

    for i, (local, rel_remote) in iterator:
        remote = f"{ftp_root}/{rel_remote}"
        try:
            status = upload_file(ftp, local, remote)
        except Exception as exc:
            status = "error"
            print(f"  ERROR [{i}/{n}] {local.name}: {exc}", file=sys.stderr)
            try:
                ftp.quit()
            except Exception:
                pass
            time.sleep(5)
            ftp = ftplib.FTP(ftp_host, timeout=60)
            ftp.login(ftp_user, ftp_pass)
            ftp.set_pasv(True)

        counts[status] += 1

        if not HAS_TQDM and i % 10 == 0:
            elapsed = time.time() - t0
            rate    = i / elapsed
            eta_s   = (n - i) / rate if rate > 0 else 0
            print(f"  [{i}/{n}]  ok={counts['ok']}  skip={counts['skip']}"
                  f"  err={counts['error']}"
                  f"  ETA {timedelta(seconds=int(eta_s))}")

    try:
        ftp.quit()
    except Exception:
        pass

    elapsed = time.time() - t0
    print(f"\nFinished in {timedelta(seconds=int(elapsed))}.")
    print(f"  Uploaded : {counts['ok']}")
    print(f"  Skipped  : {counts['skip']}")
    print(f"  Errors   : {counts['error']}")


---
## 1 — Load the real baseline manifest (what's supposed to be on EBI)

Reads `2026_Multicolour_ebi_filelist.tsv` as-is — no re-hashing, no re-scanning `Data/`.
Handles both the original header spelling (`files`) and the newer one (`filename`).


In [ ]:
def load_baseline_manifest(path: Path) -> list:
    """Read the baseline TSV manifest into a list of {filename, md5, size} dicts."""
    if not path.exists():
        raise FileNotFoundError(f"Baseline manifest not found: {path}")
    rows = []
    with path.open() as f:
        reader = csv.DictReader(f, delimiter="\t")
        name_key = "filename" if "filename" in (reader.fieldnames or []) else "files"
        for row in reader:
            rows.append({
                "filename": row[name_key],
                "md5": row["md5"],
                "size": int(row["size"]),
            })
    return rows


baseline_rows = load_baseline_manifest(BASELINE_TSV)
existing_paths = {r["filename"] for r in baseline_rows}
baseline_by_path = {r["filename"]: r for r in baseline_rows}
baseline_bytes = sum(r["size"] for r in baseline_rows)

print(f"Baseline files : {len(baseline_rows):,}")
print(f"Baseline size  : {human_size(baseline_bytes)}")

by_dir = defaultdict(int)
for r in baseline_rows:
    parts = Path(r["filename"]).parts
    key = "/".join(parts[1:3]) if len(parts) >= 3 else parts[1]
    by_dir[key] += 1

print()
for key in sorted(by_dir):
    print(f"  {key:<45} {by_dir[key]:>6}")


---
## 2 — Reconcile: get the real remote file list, upload whatever's missing

Lists every file actually present on the FTP remote (not just a count), diffs it against the
baseline manifest, and uploads exactly the files needed to close the gap — evidently uploads
that never completed back in April, independent of the folder-selectivity issue described
above. Re-run this cell at any time; it's a no-op once the remote genuinely matches the
baseline.


In [ ]:
print("Listing remote files (this walks every directory under the FTP root — can take a while) …")
remote_files_before = remote_file_set(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, EBI_PREFIX)
print(f"Remote files : {len(remote_files_before):,}")
print(f"Baseline files : {len(existing_paths):,}")

missing = existing_paths - remote_files_before
extra   = remote_files_before - existing_paths  # informational only (e.g. Checksums/*.json)

print(f"\nMissing from remote : {len(missing):,}")
print(f"On remote, not in baseline (informational, e.g. Checksums/*.json) : {len(extra):,}")

if missing:
    missing_bytes = sum(baseline_by_path[m]["size"] for m in missing)
    print(f"Total size to re-upload : {human_size(missing_bytes)}")
    print("\nFirst 10 missing:")
    for m in sorted(missing)[:10]:
        print(" ", m)


In [ ]:
def build_reconcile_pairs(smb_root: Path, ebi_prefix: str, missing: set, pairs_out: Path) -> list:
    """Build local→remote path pairs for baseline entries missing from the FTP
    remote, resolving each back to its real location under smb_root. Returns the
    filenames that couldn't be resolved locally (need manual investigation)."""
    unresolved = []
    with pairs_out.open("w") as pp:
        for fname in sorted(missing):
            rel = Path(fname).relative_to(ebi_prefix)
            local = smb_root / rel
            if not local.exists():
                unresolved.append(fname)
                continue
            pp.write(f"{local}\t{fname}\n")
    return unresolved


if missing:
    unresolved = build_reconcile_pairs(SMB_ROOT, EBI_PREFIX, missing, RECONCILE_UPLOAD_PAIRS)
    if unresolved:
        print(f"WARNING: {len(unresolved)} missing files could not be found locally under {SMB_ROOT}:")
        for f in unresolved[:20]:
            print(" ", f)
        if len(unresolved) > 20:
            print(f"  ... and {len(unresolved) - 20} more")
        print("These need manual investigation — they're in the baseline manifest but the")
        print("local file no longer exists at the expected path. Not included in the upload below.")

    run_upload(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, RECONCILE_UPLOAD_PAIRS)
else:
    print("Nothing to reconcile — remote already matches the baseline manifest.")


In [ ]:
print("Re-listing remote files to confirm reconciliation …")
remote_files_after = remote_file_set(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, EBI_PREFIX)
still_missing = existing_paths - remote_files_after

print(f"Remote files now : {len(remote_files_after):,}")
print(f"Baseline files   : {len(existing_paths):,}")

if still_missing:
    print(f"\nWARNING: {len(still_missing):,} baseline files still missing after reconciliation:")
    for m in sorted(still_missing)[:20]:
        print(" ", m)
    print("\nDo not proceed to the new-folder steps below until this is resolved —")
    print("re-run the reconciliation cells above, or investigate the errors printed during upload.")
else:
    print("\nMATCH — remote now contains every baseline file. Safe to proceed.")


---
## 3 — Hash the new folders (Massive Cells + ZWO Aureus)

Only walks `NEW_FOLDERS`, not the whole `Data/` tree, and skips anything already present in
the baseline (safe to re-run). These are large multi-GB OME-TIFs over SMB — this can take a
long time; run it in `tmux`/`screen` or leave the notebook open.


In [ ]:
def hash_new_folders(smb_root: Path, ebi_prefix: str, subfolders: list,
                      existing_paths: set) -> list:
    """Hash every TIFF under *subfolders* (relative to smb_root) that isn't already
    in *existing_paths*, returning a list of {filename, md5, size} dicts."""
    tifs = []
    for sub in subfolders:
        root = smb_root / sub
        if not root.exists():
            print(f"WARNING: folder not found, skipping: {root}")
            continue
        tifs.extend(
            p for p in root.rglob("*")
            if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
        )
    tifs.sort()

    new_tifs = [
        p for p in tifs
        if f"{ebi_prefix}/{p.relative_to(smb_root)}" not in existing_paths
    ]
    n = len(new_tifs)
    print(f"Found {len(tifs)} TIFF files under {len(subfolders)} folder(s), {n} new.")
    if n == 0:
        return []

    total_bytes = sum(p.stat().st_size for p in new_tifs)
    print(f"Total new size: {human_size(total_bytes)}")

    rows = []
    t0 = time.time()
    iterator = tqdm(enumerate(new_tifs, 1), total=n, unit="file") if HAS_TQDM \
               else enumerate(new_tifs, 1)

    with NEW_UPLOAD_PAIRS.open("w") as pp:
        for i, src in iterator:
            rel      = src.relative_to(smb_root)
            ebi_path = f"{ebi_prefix}/{rel}"
            size     = src.stat().st_size
            cksum    = md5sum(src)
            rows.append({"filename": ebi_path, "md5": cksum, "size": size})
            pp.write(f"{src}\t{ebi_path}\n")

            if not HAS_TQDM and i % 10 == 0:
                elapsed = time.time() - t0
                rate    = i / elapsed
                eta_s   = (n - i) / rate if rate > 0 else 0
                print(f"  {i}/{n}  elapsed {timedelta(seconds=int(elapsed))}"
                      f"  ETA {timedelta(seconds=int(eta_s))}")

    elapsed = time.time() - t0
    print(f"\nDone in {timedelta(seconds=int(elapsed))}.")
    print(f"Upload pairs written to {NEW_UPLOAD_PAIRS}")
    return rows


new_rows = hash_new_folders(SMB_ROOT, EBI_PREFIX, NEW_FOLDERS, existing_paths)


---
## 4 — Merge and rewrite the manifest

Backs up the current `2026_Multicolour_ebi_filelist.tsv` (as `.bak`) and then overwrites it
in place with baseline + new rows, sorted by path. This file stays the single ground-truth
manifest of everything on EBI going forward.


In [ ]:
def write_merged_manifest(rows: list, out_path: Path) -> None:
    """Write {filename, md5, size} rows to a TSV manifest, sorted by filename."""
    rows_sorted = sorted(rows, key=lambda r: r["filename"])
    with out_path.open("w") as f:
        f.write("filename\tmd5\tsize\n")
        for r in rows_sorted:
            f.write(f"{r['filename']}\t{r['md5']}\t{r['size']}\n")


if new_rows:
    backup_path = BASELINE_TSV.with_suffix(BASELINE_TSV.suffix + ".bak")
    shutil.copy2(BASELINE_TSV, backup_path)
    print(f"Backed up baseline to {backup_path}")

    merged_rows = baseline_rows + new_rows
    write_merged_manifest(merged_rows, BASELINE_TSV)

    total_bytes = sum(r["size"] for r in merged_rows)
    print(f"Merged manifest : {len(merged_rows):,} files ({human_size(total_bytes)})")
    print(f"Written to      : {BASELINE_TSV}")
else:
    merged_rows = baseline_rows
    print("No new rows to merge — manifest left unchanged.")


---
## 5 — Upload the new files to EBI (FTP)

Uploads only `NEW_UPLOAD_PAIRS` (the two new folders) — the baseline files were already
confirmed present in §2 and are not touched here. Size-based resume: re-running this cell
skips anything already uploaded at the correct size.


In [ ]:
run_upload(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, NEW_UPLOAD_PAIRS)


---
## 6 — Verify remote count now matches the fully merged manifest


In [ ]:
print("Re-listing remote files …")
remote_files_final = remote_file_set(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, EBI_PREFIX)
merged_paths = {r["filename"] for r in merged_rows}
still_missing_final = merged_paths - remote_files_final

print(f"Remote files          : {len(remote_files_final):,}")
print(f"Merged manifest files : {len(merged_rows):,}")

if not still_missing_final:
    print("MATCH — all files present on server.")
else:
    print(f"MISMATCH — {len(still_missing_final):,} files missing from server:")
    for m in sorted(still_missing_final)[:20]:
        print(" ", m)
    print("Re-run the upload cell (§5), or §2 if any of these are baseline files.")


---
## 7 — Rebuild the checksums manifest and re-upload it

Rebuilds `biostudies_checksums.json` from the full, current (merged) manifest and re-uploads
it to `Checksums/biostudies_checksums.json` on the EBI FTP area, overwriting the previous copy.


In [ ]:
def build_checksums_json(rows: list, out_path: Path, accession: str) -> dict:
    """Build biostudies_checksums.json ({path: {md5, size}}) from manifest rows."""
    files = {r["filename"]: {"md5": r["md5"], "size": r["size"]} for r in rows}
    total_bytes = sum(r["size"] for r in rows)

    manifest = {
        "_meta": {
            "description": (
                "MD5 checksums for the raw acquisition files deposited to BioStudies "
                f"{accession} (2026 Multicolour Bayer-SMLM paper) -- a curated subset of "
                "the full acquired dataset, not everything ever recorded. Computed from "
                "the original local files before upload "
                "(notebooks/EBI_Upload/EBI_Upload_Notebook.ipynb, "
                "notebooks/EBI_Upload/EBI_Upload_Update.ipynb), not re-derived from the "
                "downloaded copies, so a mismatch after download indicates real "
                "corruption/truncation."
            ),
            "accession": accession,
            "algorithm": "md5",
            "n_files": len(files),
            "total_bytes": total_bytes,
        },
        "files": files,
    }
    out_path.write_text(json.dumps(manifest, indent=2))
    print(f"Wrote {out_path}  ({len(files):,} files, {human_size(total_bytes)})")
    return manifest


def upload_checksums(ftp_host: str, ftp_user: str, ftp_pass: str, ftp_root: str,
                      local_path: Path,
                      remote_rel: str = "2026_Multicolour_Paper/Checksums/biostudies_checksums.json") -> bool:
    """Upload the checksums manifest to the EBI FTP area and verify the size matches."""
    remote = f"{ftp_root}/{remote_rel}"
    ftp = ftplib.FTP(ftp_host, timeout=60)
    ftp.login(ftp_user, ftp_pass)
    ftp.set_pasv(True)
    ensure_remote_dirs(ftp, remote)
    with local_path.open("rb") as f:
        ftp.storbinary(f"STOR {remote}", f, blocksize=1 << 20)
    ok = remote_size(ftp, remote) == local_path.stat().st_size
    ftp.quit()
    print(f"Remote path: {remote}")
    print("MATCH — checksums manifest uploaded correctly." if ok
          else "MISMATCH — re-run this cell.")
    return ok


build_checksums_json(merged_rows, CHECKSUMS_JSON, ACCESSION)
upload_checksums(FTP_HOST, FTP_USER, FTP_PASS, FTP_ROOT, CHECKSUMS_JSON)


---
## 8 — Rebuild the EBI file-list JSON (for the portal)

Writes `2026_Multicolour_Filelist.json` to `OUT_DIR` in the BioStudies file-listing schema,
covering every file in the merged manifest plus the checksums manifest itself. **Not**
uploaded automatically — submit it through the BioImage Archive portal by hand.


In [ ]:
def build_ebi_filelist_json(rows: list, checksums_path: Path,
                             ebi_prefix: str, out_path: Path) -> list:
    """Build the flat BioStudies file-listing JSON from manifest rows, plus one
    entry for the checksums manifest itself."""
    entries = []
    for r in sorted(rows, key=lambda r: r["filename"]):
        entries.append({
            "path": r["filename"],
            "size": r["size"],
            "attributes": [{"name": "size", "value": str(r["size"])}],
            "type": "file",
        })

    cksum_size = checksums_path.stat().st_size
    entries.append({
        "path": f"{ebi_prefix}/Checksums/{checksums_path.name}",
        "size": cksum_size,
        "attributes": [{"name": "size", "value": str(cksum_size)}],
        "type": "file",
    })

    out_path.write_text(json.dumps(entries, indent=2))
    print(f"Wrote {out_path}  ({len(entries):,} entries)")
    return entries


build_ebi_filelist_json(merged_rows, CHECKSUMS_JSON, EBI_PREFIX, FILELIST_JSON)


---
## Checklist

- [ ] SMB share mounted at the path in §1
- [ ] Baseline manifest loaded (§1) — 1,342 files, matches `biostudies_checksums.json`'s
      recorded `n_files`/`total_bytes`
- [ ] Remote file list retrieved and diffed against baseline (§2)
- [ ] Missing baseline files re-uploaded and reconciliation re-verified (§2) — 0 still missing
- [ ] New folders hashed (§3): `ZWO/20260623_MASSIVECELLS`, `ZWO/20260624_SAureus_NR4A`
- [ ] Manifest merged and backed up (§4) — `2026_Multicolour_ebi_filelist.tsv.bak` written
- [ ] New files uploaded (§5), errors = 0
- [ ] Remote count matches merged manifest (§6)
- [ ] `biostudies_checksums.json` rebuilt and re-uploaded (§7)
- [ ] `2026_Multicolour_Filelist.json` regenerated (§8)
- [ ] Updated `2026_Multicolour_Filelist.json` submitted through the BioImage Archive portal
- [ ] Stale files from the 2026-08-28 run removed: `EBI_Upload/ebi_filelist.tsv`,
      `EBI_Upload/ebi_upload_paths.txt`
